# AutoSDE team workbench

Validate the coordinator, specialists and aggregator with the **same code and prompt files** the CLI and web page use.

- Prompts are in `autosde/prompts/` (`coordinator.md`, `specialist.md`, `aggregator.md`, `focus/<agent>.md`). They are re-read on every call: edit, then re-run a cell.
- Section 2 is free (a scripted model). Section 3 calls OpenAI and is off until you opt in.
- Reviews from section 3 are stored in the same SQLite database as `autosde review`, so section 4 and `autosde reviews` see them.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))

from autosde import report
from autosde.config import load_settings
from autosde.db import Store
from autosde.git import ChangedFile
from autosde.llm import OpenAIModel, ScriptedModel
from autosde.review import run_review
from autosde.team import AGENTS, PROMPTS, Team, prompt

settings = load_settings()
print("prompts:", PROMPTS)
print("agents:", ", ".join(AGENTS))
print("database:", settings.db_path)
print("gitleaks:", settings.gitleaks or "not found (set AUTOSDE_GITLEAKS)")

## 1. The prompts

Exactly what each role receives as its system prompt.

In [ ]:
print(prompt("coordinator.md"))
print("-" * 60)
print(prompt("specialist.md", agent="python", focus="loop totals"))
print("-" * 60)
print(prompt("aggregator.md"))

## 2. Free scripted team run

A `ScriptedModel` replies per role, so you can check planning, routing, scope rules, line clamping and aggregation without calling a model. Change the script to test edge cases (a missing coordinator reply, a finding for a file the specialist does not own, an aggregator rejection, ...).

In [ ]:
head = 'def calculate_total(items):\n    total = 0\n    for item in items:\n        total = item["price"]\n    return total\n'
files = [
    ChangedFile(
        path="calculator.py",
        status="M",
        head=head,
        diff="@@ -1,2 +1,5 @@\n-    return sum(i[\"price\"] for i in items)\n+    total = 0\n+    for item in items:\n+        total = item[\"price\"]\n+    return total\n",
    )
]
finding = {
    "file": "calculator.py", "start_line": 4, "end_line": 4, "severity": "medium",
    "category": "correctness", "title": "Total is overwritten",
    "description": "The loop assigns instead of adding.", "recommendation": "Use +=.",
    "confidence": 0.9,
}
scripted = ScriptedModel({
    "coordinator": [{"assignments": [{"agent": "python", "files": ["calculator.py"], "focus": "loop totals"}],
                     "rationale": "one python file"}],
    "python": [{"findings": [finding, dict(finding, title="Same bug again")], "summary": "ok"}],
    "aggregator": [{"rejected": [{"id": "python-2", "reason": "duplicate"}], "summary": "One real defect."}],
})
result = Team(scripted).run(files)

print("plan:", result.plan)
for call in result.calls:
    print(f"{call.role:<11} {call.agent:<11} {call.status:<6} files={call.files} error={call.error}")
for f in result.findings:
    print(f"{f.id:<10} kept={f.kept!s:<5} {f.file}:{f.start_line} {f.title} {f.rejection or ''}")
print("limitations:", result.limitations)
print("summary:", result.summary)

In [ ]:
# What each role was sent (first 700 characters):
for call in scripted.calls:
    print("===", call["role"])
    print(call["prompt"][:700])

## 3. Real team run on a commit range (paid)

Start Jupyter with `OPENAI_API_KEY` already in its environment; never paste keys into cells. The range is read from committed history only. Each file is scanned with gitleaks before anything is sent, and source code is sent to OpenAI. A typical small review costs well under a cent with the default model.

In [ ]:
AUTHORIZE_REPOSITORY_READ = False  # set True to read the repository
ALLOW_PAID_INFERENCE = False       # set True to call OpenAI

REPO = pathlib.Path.home() / "Documents/ChatGPT/calculator"
RANGE = "HEAD~1..HEAD"  # A..B, A.. or a single commit

if AUTHORIZE_REPOSITORY_READ and ALLOW_PAID_INFERENCE:
    store = Store(settings.db_path)
    model = OpenAIModel.from_settings(settings)
    review_id = run_review(REPO, RANGE, store=store, model=model, settings=settings)
    print(report.text(store.review(review_id)))
else:
    print("Skipped: set both opt-ins to True to run a paid review.")

## 4. Query the database

Projects, reviews and which agents ran. The same data is behind `autosde projects`, `autosde reviews`, `autosde agents` and the web page.

In [ ]:
store = Store(settings.db_path)
for p in store.projects():
    print(p["id"], p["name"], p["reviews"], "reviews", p["path"])
print()
for r in store.reviews(limit=10):
    print(r["id"], r["project"], r["range_spec"], r["status"], r["findings"], "findings", r["agents"])
print()
for a in store.agents():
    print(a["project"], a["role"], a["agent"], a["runs"], "runs", a["failed"], "failed", a["findings_kept"], "kept")

## 5. Iterate

1. Edit a prompt in `autosde/prompts/`.
2. Re-run section 2 (free) or 3 (paid).
3. Run the tests: `python3 -m unittest discover -s tests -t tests`. Commit prompt changes with notebook outputs cleared.